# Produce data

- Mô phỏng quá trình generate và đẩy dữ liệu vào db của một cty
- Mỗi giây sẽ chèn tầm 17_000 record vào db

In [1]:
import sys
sys.path.append('/home/longnb32/FPT/ops/ml-ops/src')

import time
import numpy as np
import pandas as pd
from utils.db_connection import PostgresConnection
import s3fs
import os

from pyspark.sql import SparkSession, DataFrame, Window
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType, StringType, TimestampType

from dotenv import load_dotenv

load_dotenv('../../../../.env_file/infra.env'), load_dotenv('../../../../.env_file/app.env')

(True, True)

In [3]:
spark = (
    SparkSession
    .builder
    .appName('produce data')
    .config('spark.jars', './spark/jar/postgresql-42.7.11.jar')
    .config("spark.jars.packages", "org.apache.hadoop:hadoop-aws:3.4.2,software.amazon.awssdk:bundle:2.29.38")
    .config("spark.hadoop.fs.s3a.access.key", os.getenv('AWS_ACCESS_KEY_ID'))
    .config("spark.hadoop.fs.s3a.secret.key", os.getenv('AWS_SECRET_ACCESS_KEY'))
    .getOrCreate()
)
spark

:: loading settings :: url = jar:file:/home/longnb32/FPT/ops/ml-ops/.venv/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /home/longnb32/.ivy2.5.2/cache
The jars for the packages stored in: /home/longnb32/.ivy2.5.2/jars
org.apache.hadoop#hadoop-aws added as a dependency
software.amazon.awssdk#bundle added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-8b43c78f-9be0-4493-9093-508086f19bd3;1.0
	confs: [default]
	found org.apache.hadoop#hadoop-aws;3.4.2 in central
	found software.amazon.awssdk#bundle;2.29.52 in central
	found software.amazon.s3.analyticsaccelerator#analyticsaccelerator-s3;1.2.1 in central
	found org.wildfly.openssl#wildfly-openssl;2.1.4.Final in central
:: resolution report :: resolve 315ms :: artifacts dl 11ms
	:: modules in use:
	org.apache.hadoop#hadoop-aws;3.4.2 from central in [default]
	org.wildfly.openssl#wildfly-openssl;2.1.4.Final from central in [de

- Thông tin cột, kiểu dữ liệu của cột, mapping để đổi tên cột

In [11]:
LIST_COLS = [
    "vendor_id", "pickup_datetime", "dropoff_datetime", "passenger_count",
    "trip_distance", "rate_code_id", "store_and_fwd_flag", "pickup_longitude",
    "pickup_latitude", "dropoff_longitude", "dropoff_latitude", "pu_location_id",
    "do_location_id", "payment_type", "fare_amount", "extra", "mta_tax",
    "surcharge", "improvement_surcharge", "tip_amount", "tolls_amount",
    "total_amount", "congestion_surcharge", "airport_fee", "cbd_congestion_fee",
]

LIST_DTYPES = {
    "vendor_id":             "object",
    "pickup_datetime":       "datetime64[ns]",
    "dropoff_datetime":      "datetime64[ns]",
    "passenger_count":       "float64",
    "trip_distance":         "float64",
    "rate_code_id":          "float64",
    "store_and_fwd_flag":    "object",
    "pickup_longitude":      "float64",
    "pickup_latitude":       "float64",
    "dropoff_longitude":     "float64",
    "dropoff_latitude":      "float64",
    "pu_location_id":        "object",
    "do_location_id":        "object",
    "payment_type":          "object",
    "fare_amount":           "float64",
    "extra":                 "float64",
    "mta_tax":               "float64",
    "surcharge":             "float64",
    "improvement_surcharge": "float64",
    "tip_amount":            "float64",
    "tolls_amount":          "float64",
    "total_amount":          "float64",
    "congestion_surcharge":  "float64",
    "airport_fee":           "float64",
    "cbd_congestion_fee":    "float64",
}

map1 = {
    "columns": 'VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|airport_fee',
    'mapping': {
        "VendorID": "vendor_id",
        "tpep_pickup_datetime": "pickup_datetime",
        "tpep_dropoff_datetime": "dropoff_datetime",
        "RatecodeID": "rate_code_id",
        "PULocationID": "pu_location_id",
        "DOLocationID": "do_location_id",
        "payment_type": "payment_type",
        "fare_amount": "fare_amount",
    }
}

map2 = {
    'columns':  'VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee',
    'mapping': {
        "VendorID": "vendor_id",
        "tpep_pickup_datetime": "pickup_datetime",
        "tpep_dropoff_datetime": "dropoff_datetime",
        "RatecodeID": "rate_code_id",
        "PULocationID": "pu_location_id",
        "DOLocationID": "do_location_id",
        "Airport_fee": "airport_fee",
    }
}

map3 = {
    'columns': 'vendor_name|Trip_Pickup_DateTime|Trip_Dropoff_DateTime|Passenger_Count|Trip_Distance|Start_Lon|Start_Lat|Rate_Code|store_and_forward|End_Lon|End_Lat|Payment_Type|Fare_Amt|surcharge|mta_tax|Tip_Amt|Tolls_Amt|Total_Amt',
    'mapping': {
        "vendor_name": "vendor_id",
        "Trip_Pickup_DateTime": "pickup_datetime",
        "Trip_Dropoff_DateTime": "dropoff_datetime",
        "Passenger_Count": "passenger_count",
        "Trip_Distance": "trip_distance",
        "Start_Lon": "pickup_longitude",
        "Start_Lat": "pickup_latitude",
        "Rate_Code": "rate_code_id",
        "store_and_forward": "store_and_fwd_flag",
        "End_Lon": "dropoff_longitude",
        "End_Lat": "dropoff_latitude",
        "Payment_Type": "payment_type",
        "Fare_Amt": "fare_amount",
        "Tip_Amt": "tip_amount",
        "Tolls_Amt": "tolls_amount",
        "Total_Amt": "total_amount",
    }
}

map4 = {
    'columns': 'VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee',
    'mapping': {
        "VendorID": "vendor_id",
        "tpep_pickup_datetime": "pickup_datetime",
        "tpep_dropoff_datetime": "dropoff_datetime",
        "RatecodeID": "rate_code_id",
        "PULocationID": "pu_location_id",
        "DOLocationID": "do_location_id",
        "Airport_fee": "airport_fee",
    }
}

map5 = {
    'columns':  'vendor_id|pickup_datetime|dropoff_datetime|passenger_count|trip_distance|pickup_longitude|pickup_latitude|rate_code|store_and_fwd_flag|dropoff_longitude|dropoff_latitude|payment_type|fare_amount|surcharge|mta_tax|tip_amount|tolls_amount|total_amount',
    'mapping': {
        "rate_code": "rate_code_id",
    }
}

map6 = {
    'columns':  'vendorID|tpepPickupDateTime|tpepDropoffDateTime|passengerCount|tripDistance|puLocationId|doLocationId|startLon|startLat|endLon|endLat|rateCodeId|storeAndFwdFlag|paymentType|fareAmount|extra|mtaTax|improvementSurcharge|tipAmount|tollsAmount|totalAmount',
    'mapping': {
        "vendorID": "vendor_id",
        "tpepPickupDateTime": "pickup_datetime",
        "tpepDropoffDateTime": "dropoff_datetime",
        "passengerCount": "passenger_count",
        "tripDistance": "trip_distance",
        "puLocationId": "pu_location_id",
        "doLocationId": "do_location_id",
        "startLon": "pickup_longitude",
        "startLat": "pickup_latitude",
        "endLon": "dropoff_longitude",
        "endLat": "dropoff_latitude",
        "rateCodeId": "rate_code_id",
        "storeAndFwdFlag": "store_and_fwd_flag",
        "paymentType": "payment_type",
        "fareAmount": "fare_amount",
        "mtaTax": "mta_tax",
        "improvementSurcharge": "improvement_surcharge",
        "tipAmount": "tip_amount",
        "tollsAmount": "tolls_amount",
        "totalAmount": "total_amount",
    }
}

map7 = {
    'columns': 'vendor_id|pickup_datetime|dropoff_datetime|passenger_count|trip_distance|pickup_longitude|pickup_latitude|rate_code|store_and_fwd_flag|dropoff_longitude|dropoff_latitude|payment_type|fare_amount|surcharge|mta_tax|tip_amount|tolls_amount|total_amount|__index_level_0__',
    'mapping': {
        "rate_code": "rate_code_id",
    }
}

SCHEMA_MAPPINGS = [map1, map2, map3, map4, map5, map6, map7]

- Tạo bảng

In [5]:
def create_table(conn):
    """Tạo bảng nyc_taxi nếu chưa tồn tại."""
    query = """
    CREATE TABLE IF NOT EXISTS public.nyc_taxi (
        vendor_id               TEXT,
        pickup_datetime         TIMESTAMP,
        dropoff_datetime        TIMESTAMP,
        passenger_count         FLOAT,
        trip_distance           FLOAT,
        rate_code_id            FLOAT,
        store_and_fwd_flag      TEXT,
        pickup_longitude        FLOAT,
        pickup_latitude         FLOAT,
        dropoff_longitude       FLOAT,
        dropoff_latitude        FLOAT,
        pu_location_id          TEXT,
        do_location_id          TEXT,
        payment_type            TEXT,
        fare_amount             FLOAT,
        extra                   FLOAT,
        mta_tax                 FLOAT,
        surcharge               FLOAT,
        improvement_surcharge   FLOAT,
        tip_amount              FLOAT,
        tolls_amount            FLOAT,
        total_amount            FLOAT,
        congestion_surcharge    FLOAT,
        airport_fee             FLOAT,
        cbd_congestion_fee      FLOAT
    );
    """
    with conn.cursor() as cur:
        cur.execute(query)
    conn.commit()
    print("Table 'nyc_taxi' created successfully.")

- Transform

In [6]:
SPARK_DTYPES = {
    "object": StringType(),
    "float64": DoubleType(),
    "datetime64[ns]": TimestampType(),
}

def _resolve_mapping(df_columns: str) -> dict:
    """Tìm schema mapping khớp với columns của file."""
    for schema in SCHEMA_MAPPINGS:
        if schema["columns"] == df_columns:
            return schema["mapping"]
    raise ValueError(f"No mapping found for {df_columns}")

def transform_data(df: DataFrame) -> DataFrame:
    """
    1. Đổi tên cột theo mapping tương ứng.
    2. Thêm cột canonical còn thiếu (fill null).
    3. Đổi kiểu dữ liệu về canonical dtype.
    4. Chỉ giữ lại các cột canonical.
    """
    # 1. Resolve & rename
    mapping = _resolve_mapping('|'.join(df.columns))
    for old_name, new_name in mapping.items():
        df = df.withColumnRenamed(old_name, new_name)

    # 2. Thêm cột còn thiếu
    for col in LIST_COLS:
        if col not in df.columns:
            df = df.withColumn(col, F.lit(None))

    # 3. Cast dtype
    for col, dtype in LIST_DTYPES.items():
        spark_type = SPARK_DTYPES[dtype]
        df = df.withColumn(col, F.col(col).cast(spark_type))

    # 4. Chỉ giữ các cột canonical
    return df.select(LIST_COLS)

---
- Read data and transform

In [12]:
# read parquet
filepath = "s3a://flezi-data/nyc-taxi/yellow_tripdata_2010-02.parquet"
df:DataFrame = spark.read.parquet(filepath)

# get 140_000 samples (asc pickup-date)
df = transform_data(df.sample(min(140_000/df.count(), 1)))
df.show(10)


26/06/09 08:44:33 INFO HadoopFSUtils: Listing s3a://flezi-data/nyc-taxi/yellow_tripdata_2010-02.parquet with listFiles API
26/06/09 08:44:33 INFO InMemoryFileIndex: It took 42 ms to list leaf files for 1 paths.
26/06/09 08:44:33 INFO SparkContext: Starting job: parquet at NativeMethodAccessorImpl.java:0
26/06/09 08:44:33 INFO DAGScheduler: Got job 7 (parquet at NativeMethodAccessorImpl.java:0) with 1 output partitions
26/06/09 08:44:33 INFO DAGScheduler: Final stage: ResultStage 8 (parquet at NativeMethodAccessorImpl.java:0)
26/06/09 08:44:33 INFO DAGScheduler: Parents of final stage: List()
26/06/09 08:44:33 INFO DAGScheduler: Missing parents: List()
26/06/09 08:44:33 INFO DAGScheduler: Missing parents found for ResultStage 8: List()
26/06/09 08:44:33 INFO DAGScheduler: Submitting ResultStage 8 (MapPartitionsRDD[18] at parquet at NativeMethodAccessorImpl.java:0), which has no missing parents
26/06/09 08:44:33 INFO MemoryStore: Block broadcast_10 stored as values in memory (estimated s

+---------+-------------------+-------------------+---------------+------------------+------------+------------------+------------------+---------------+------------------+----------------+--------------+--------------+------------+-----------------+-----+-------+---------+---------------------+----------+------------+------------+--------------------+-----------+------------------+
|vendor_id|    pickup_datetime|   dropoff_datetime|passenger_count|     trip_distance|rate_code_id|store_and_fwd_flag|  pickup_longitude|pickup_latitude| dropoff_longitude|dropoff_latitude|pu_location_id|do_location_id|payment_type|      fare_amount|extra|mta_tax|surcharge|improvement_surcharge|tip_amount|tolls_amount|total_amount|congestion_surcharge|airport_fee|cbd_congestion_fee|
+---------+-------------------+-------------------+---------------+------------------+------------+------------------+------------------+---------------+------------------+----------------+--------------+--------------+---------

26/06/09 08:44:36 INFO CodecPool: Got brand-new decompressor [.gz]
26/06/09 08:44:36 INFO Executor: Finished task 2.0 in stage 13.0 (TID 33). 3788 bytes result sent to driver
26/06/09 08:44:36 INFO TaskSetManager: Finished task 2.0 in stage 13.0 (TID 33) in 1552 ms on ip-10-0-0-6.ap-northeast-1.compute.internal (executor driver) (4/4)
26/06/09 08:44:36 INFO TaskSchedulerImpl: Removed TaskSet 13.0 whose tasks have all completed, from pool 
26/06/09 08:44:36 INFO DAGScheduler: ResultStage 13 (showString at NativeMethodAccessorImpl.java:0) finished in 1559 ms
26/06/09 08:44:36 INFO DAGScheduler: Job 11 is finished. Cancelling potential speculative or zombie tasks for this job
26/06/09 08:44:36 INFO TaskSchedulerImpl: Canceling stage 13
26/06/09 08:44:36 INFO TaskSchedulerImpl: Killing all running tasks in stage 13: Stage finished
26/06/09 08:44:36 INFO DAGScheduler: Job 11 finished: showString at NativeMethodAccessorImpl.java:0, took 1561.144969 ms
26/06/09 08:44:36 INFO CodeGenerator: Co

- Để có thể sort thì cần phải đánh index cho data
- Để có thể đánh index thì phải gom hết data vào 1 partition. Việc này nguy hiểm nếu dữ liệu quá lớn
- Tuy nhiên, đây chỉ là mô phỏng quá trình đẩy dữ liệu từ từ vào db, và dữ liệu cũng không quá lớn nên ok

In [7]:
batch_size = 17_000

# add an index column
window_spec = Window.orderBy('pickup_datetime')
df = df.withColumn('index', F.row_number().over(window_spec) - 1)
df.createOrReplaceTempView('nyc_taxi')

total_rows = df.count()
total_rows

26/05/20 09:29:48 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


140018

In [8]:
df.show(10)

26/05/20 09:29:55 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/20 09:29:55 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/20 09:29:55 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


+---------+-------------------+-------------------+---------------+------------------+------------+------------------+------------------+---------------+------------------+----------------+--------------+--------------+------------+-----------+-----+-------+---------+---------------------+----------+------------+-----------------+--------------------+-----------+------------------+-----+
|vendor_id|    pickup_datetime|   dropoff_datetime|passenger_count|     trip_distance|rate_code_id|store_and_fwd_flag|  pickup_longitude|pickup_latitude| dropoff_longitude|dropoff_latitude|pu_location_id|do_location_id|payment_type|fare_amount|extra|mta_tax|surcharge|improvement_surcharge|tip_amount|tolls_amount|     total_amount|congestion_surcharge|airport_fee|cbd_congestion_fee|index|
+---------+-------------------+-------------------+---------------+------------------+------------+------------------+------------------+---------------+------------------+----------------+--------------+--------------

In [11]:
for i in range(17000, total_rows, batch_size):
    start_index = i
    end_index = min(i+batch_size, total_rows-1)
    print(f'Process from {start_index} to {end_index}')

    tmp_df = spark.sql(f'''
        select *
        from nyc_taxi
        where (nyc_taxi.index >= {start_index}) and (nyc_taxi.index < {end_index})
    ''')
    tmp_df.drop('index').write \
        .format("jdbc") \
        .option("url", "jdbc:postgresql://localhost:5432/company") \
        .option("dbtable", "nyc_taxi") \
        .option("user", "company") \
        .option("password", "company") \
        .option("driver", "org.postgresql.Driver") \
        .mode("append") \
        .save()
    break

Process from 17000 to 34000


26/05/20 09:31:54 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/20 09:31:54 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/20 09:31:54 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/20 09:32:02 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/20 09:32:02 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


In [13]:
pd_df = tmp_df.toPandas()
pd_df.shape

26/05/20 09:32:09 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/20 09:32:09 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/20 09:32:09 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/20 09:32:16 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/20 09:32:16 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


(17000, 26)

In [14]:
pd_df.head()

,vendor_id,pickup_datetime,dropoff_datetime,passenger_count,trip_distance,rate_code_id,store_and_fwd_flag,pickup_longitude,pickup_latitude,dropoff_longitude,...,mta_tax,surcharge,improvement_surcharge,tip_amount,tolls_amount,total_amount,congestion_surcharge,airport_fee,cbd_congestion_fee,index
0,VTS,2010-01-05 13:43:00,2010-01-05 13:55:00,1.0,0.75,1.0,None,-73.968263,40.762598,-73.977300,...,0.5,0.0,NaN,0.00,0.0,7.80,NaN,NaN,NaN,17000
1,CMT,2010-01-05 13:43:17,2010-01-05 13:47:23,1.0,0.80,1.0,0,-73.993428,40.727361,-73.987621,...,0.5,0.0,NaN,0.75,0.0,5.75,NaN,NaN,NaN,17001
2,DDS,2010-01-05 13:43:30,2010-01-05 13:49:04,2.0,0.80,1.0,None,-73.971685,40.760172,-73.992723,...,-1.0,0.0,NaN,0.00,0.0,5.40,NaN,NaN,NaN,17002
3,CMT,2010-01-05 13:43:53,2010-01-05 13:50:04,3.0,1.00,1.0,0,-73.994666,40.741344,-73.988374,...,0.5,0.0,NaN,0.00,0.0,5.80,NaN,NaN,NaN,17003
4,VTS,2010-01-05 13:44:00,2010-01-05 14:23:00,1.0,17.58,1.0,None,-73.987490,40.737768,-73.782715,...,0.5,0.0,NaN,0.00,5.5,51.00,NaN,NaN,NaN,17004


In [16]:
tmp_df.show()

+---------+-------------------+-------------------+---------------+------------------+------------+------------------+------------------+---------------+------------------+----------------+--------------+--------------+------------+-----------------+-----+-------+---------+---------------------+----------+------------+-----------------+--------------------+-----------+------------------+-----+
|vendor_id|    pickup_datetime|   dropoff_datetime|passenger_count|     trip_distance|rate_code_id|store_and_fwd_flag|  pickup_longitude|pickup_latitude| dropoff_longitude|dropoff_latitude|pu_location_id|do_location_id|payment_type|      fare_amount|extra|mta_tax|surcharge|improvement_surcharge|tip_amount|tolls_amount|     total_amount|congestion_surcharge|airport_fee|cbd_congestion_fee|index|
+---------+-------------------+-------------------+---------------+------------------+------------+------------------+------------------+---------------+------------------+----------------+--------------+--